# NOTEBOOK 3: ANÁLISIS DESCRIPTIVO

---

## Proyecto: Arquitectura de BI y Big Data para Análisis del Turismo Académico en Medellín

**Objetivo del Notebook:** Realizar análisis estadístico descriptivo completo de los datos de movilidad estudiantil, identificar patrones, tendencias y generar insights clave para la toma de decisiones.

---

### Contenido:
1. Carga de datos limpios
2. Estadísticas descriptivas generales
3. Análisis temporal (tendencias y estacionalidad)
4. Análisis geográfico (países y regiones)
5. Análisis por tipo de movilidad
6. Análisis financiero
7. Identificación de outliers
8. Correlaciones y asociaciones
9. Síntesis de hallazgos

---
## 1. CONFIGURACIÓN E IMPORTACIÓN

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os
from scipy import stats

# Configuración de visualizaciones
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Rutas
BASE_DIR = Path(os.getcwd())
DATA_PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
OUTPUTS_DIR = BASE_DIR / 'outputs' / 'graficas'
REPORTS_DIR = BASE_DIR / 'outputs' / 'reportes'
RESULTS_DIR = BASE_DIR / 'data' / 'results'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Configuración completada")

---
## 2. CARGA DE DATOS LIMPIOS

In [ ]:
# Cargar datos procesados del Notebook 2
df = pd.read_csv(DATA_PROCESSED_DIR / 'datos_consolidados_limpios.csv')

print("="*80)
print("DATOS CARGADOS PARA ANÁLISIS DESCRIPTIVO")
print("="*80)
print(f"\nRegistros: {len(df):,}")
print(f"Columnas: {len(df.columns)}")
print(f"\nUniversidades: {df['UNIVERSIDAD'].value_counts().to_dict()}")
print(f"\nPrimeras filas:")
display(df.head())

---
## 3. ESTADÍSTICAS DESCRIPTIVAS GENERALES

In [ ]:
print("\n" + "="*80)
print("ESTADÍSTICAS DESCRIPTIVAS GENERALES")
print("="*80)

# Resumen estadístico de variables numéricas
print("\n1. VARIABLES NUMÉRICAS")
print("-" * 80)
estadisticas_numericas = df[['NUM_DIAS_MOVILIDAD', 'FINANCIACION_TOTAL']].describe()
display(estadisticas_numericas.T)

# Resumen de variables categóricas
print("\n2. VARIABLES CATEGÓRICAS PRINCIPALES")
print("-" * 80)

categoricas = ['PAIS_EXTRANJERO', 'TIPO_MOV_EST_EXTRANJ', 'CATEGORIA_DURACION', 'UNIVERSIDAD']

resumen_categorico = pd.DataFrame({
    'Variable': categoricas,
    'Categorías_Únicas': [df[col].nunique() if col in df.columns else 0 for col in categoricas],
    'Moda': [df[col].mode()[0] if col in df.columns and len(df[col].mode()) > 0 else 'N/A' for col in categoricas],
    'Frecuencia_Moda': [df[col].value_counts().iloc[0] if col in df.columns else 0 for col in categoricas]
})

display(resumen_categorico)

# Métricas clave
print("\n3. MÉTRICAS CLAVE DEL PROYECTO")
print("-" * 80)
print(f"Total de estudiantes internacionales registrados: {len(df):,}")
print(f"Duración promedio de movilidad: {df['NUM_DIAS_MOVILIDAD'].mean():.1f} días")
print(f"Duración mediana: {df['NUM_DIAS_MOVILIDAD'].median():.1f} días")
print(f"Financiación total promedio: ${df['FINANCIACION_TOTAL'].mean():,.0f} COP")
print(f"Países de origen representados: {df['PAIS_EXTRANJERO'].nunique()}")
print(f"Instituciones extranjeras: {df['INSTITUCION_EXTRANJERA'].nunique()}")

---
## 4. ANÁLISIS TEMPORAL

Análisis de tendencias y patrones temporales en la movilidad estudiantil.

In [ ]:
print("\n" + "="*80)
print("ANÁLISIS TEMPORAL")
print("="*80)

# Crear figura con subplots
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Análisis Temporal de Movilidad Estudiantil', fontsize=16, fontweight='bold')

# 1. Tendencia anual
movilidad_anual = df.groupby('AÑO').size()
axes[0, 0].plot(movilidad_anual.index, movilidad_anual.values, marker='o', linewidth=2, markersize=8, color='steelblue')
axes[0, 0].fill_between(movilidad_anual.index, movilidad_anual.values, alpha=0.3, color='steelblue')
axes[0, 0].set_title('Evolución Anual de Movilidad Estudiantil', fontweight='bold')
axes[0, 0].set_xlabel('Año')
axes[0, 0].set_ylabel('Número de Estudiantes')
axes[0, 0].grid(True, alpha=0.3)
for i, v in enumerate(movilidad_anual.values):
    axes[0, 0].text(movilidad_anual.index[i], v + 0.5, str(v), ha='center', fontweight='bold')

# 2. Distribución por semestre
movilidad_semestre = df.groupby(['AÑO', 'SEMESTRE']).size().reset_index(name='Cantidad')
movilidad_semestre['Periodo'] = movilidad_semestre['AÑO'].astype(str) + '-' + movilidad_semestre['SEMESTRE'].astype(str)
axes[0, 1].bar(range(len(movilidad_semestre)), movilidad_semestre['Cantidad'], 
               color=['coral' if s == 1 else 'skyblue' for s in movilidad_semestre['SEMESTRE']])
axes[0, 1].set_title('Movilidad por Periodo (Año-Semestre)', fontweight='bold')
axes[0, 1].set_xlabel('Periodo')
axes[0, 1].set_ylabel('Número de Estudiantes')
axes[0, 1].set_xticks(range(len(movilidad_semestre)))
axes[0, 1].set_xticklabels(movilidad_semestre['Periodo'], rotation=45, ha='right')
axes[0, 1].legend(['Semestre 1', 'Semestre 2'])

# 3. Comparación semestral (promedio)
comparacion_semestral = df.groupby('SEMESTRE').size()
axes[1, 0].pie(comparacion_semestral.values, labels=['Semestre 1', 'Semestre 2'], 
               autopct='%1.1f%%', startangle=90, colors=['coral', 'skyblue'])
axes[1, 0].set_title('Distribución General por Semestre', fontweight='bold')

# 4. Tasa de crecimiento interanual
tasa_crecimiento = movilidad_anual.pct_change() * 100
axes[1, 1].bar(tasa_crecimiento.index[1:], tasa_crecimiento.values[1:], 
               color=['green' if x > 0 else 'red' for x in tasa_crecimiento.values[1:]])
axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[1, 1].set_title('Tasa de Crecimiento Interanual (%)', fontweight='bold')
axes[1, 1].set_xlabel('Año')
axes[1, 1].set_ylabel('Crecimiento (%)')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / '03_analisis_temporal.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Gráfica guardada: {OUTPUTS_DIR / '03_analisis_temporal.png'}")
plt.show()

# Estadísticas temporales
print("\nESTADÍSTICAS TEMPORALES:")
print("-" * 80)
print(f"Año con mayor movilidad: {movilidad_anual.idxmax()} ({movilidad_anual.max()} estudiantes)")
print(f"Año con menor movilidad: {movilidad_anual.idxmin()} ({movilidad_anual.min()} estudiantes)")
print(f"Tasa de crecimiento promedio anual: {tasa_crecimiento.mean():.2f}%")

---
## 5. ANÁLISIS GEOGRÁFICO

Análisis de la distribución geográfica de los estudiantes internacionales.

In [ ]:
print("\n" + "="*80)
print("ANÁLISIS GEOGRÁFICO")
print("="*80)

# Top países
top_paises = df['PAIS_EXTRANJERO'].value_counts().head(15)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Análisis Geográfico de Movilidad Estudiantil', fontsize=16, fontweight='bold')

# 1. Top 15 países de origen
top_paises.plot(kind='barh', ax=axes[0, 0], color='teal')
axes[0, 0].set_title('Top 15 Países de Origen de Estudiantes', fontweight='bold')
axes[0, 0].set_xlabel('Número de Estudiantes')
axes[0, 0].set_ylabel('País')
axes[0, 0].invert_yaxis()

# 2. Distribución porcentual (top 10)
top_10_paises = df['PAIS_EXTRANJERO'].value_counts().head(10)
otros = df['PAIS_EXTRANJERO'].value_counts()[10:].sum()
distribucion_paises = pd.concat([top_10_paises, pd.Series({'Otros': otros})])

axes[0, 1].pie(distribucion_paises.values, labels=distribucion_paises.index, 
               autopct='%1.1f%%', startangle=140)
axes[0, 1].set_title('Distribución Porcentual por País (Top 10 + Otros)', fontweight='bold')

# 3. Evolución temporal de top 5 países
top_5_paises = df['PAIS_EXTRANJERO'].value_counts().head(5).index
for pais in top_5_paises:
    df_pais = df[df['PAIS_EXTRANJERO'] == pais]
    evolucion = df_pais.groupby('AÑO').size()
    axes[1, 0].plot(evolucion.index, evolucion.values, marker='o', label=pais, linewidth=2)

axes[1, 0].set_title('Evolución Temporal - Top 5 Países', fontweight='bold')
axes[1, 0].set_xlabel('Año')
axes[1, 0].set_ylabel('Número de Estudiantes')
axes[1, 0].legend(loc='best')
axes[1, 0].grid(True, alpha=0.3)

# 4. Duración promedio de estadía por país (top 10)
duracion_por_pais = df.groupby('PAIS_EXTRANJERO')['NUM_DIAS_MOVILIDAD'].mean().nlargest(10)
duracion_por_pais.plot(kind='bar', ax=axes[1, 1], color='orange')
axes[1, 1].set_title('Duración Promedio de Estadía por País (Top 10)', fontweight='bold')
axes[1, 1].set_xlabel('País')
axes[1, 1].set_ylabel('Días Promedio')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / '03_analisis_geografico.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Gráfica guardada: {OUTPUTS_DIR / '03_analisis_geografico.png'}")
plt.show()

# Estadísticas geográficas
print("\nESTADÍSTICAS GEOGRÁFICAS:")
print("-" * 80)
print(f"Total de países representados: {df['PAIS_EXTRANJERO'].nunique()}")
print(f"\nTop 5 países:")
for i, (pais, cantidad) in enumerate(top_paises.head(5).items(), 1):
    porcentaje = (cantidad / len(df)) * 100
    print(f"  {i}. {pais}: {cantidad} estudiantes ({porcentaje:.1f}%)")

# Concentración geográfica
top_3_concentracion = (top_paises.head(3).sum() / len(df)) * 100
print(f"\nConcentración Top 3 países: {top_3_concentracion:.1f}%")

---
## 6. ANÁLISIS POR TIPO DE MOVILIDAD

In [ ]:
print("\n" + "="*80)
print("ANÁLISIS POR TIPO DE MOVILIDAD")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Análisis por Tipo de Movilidad Estudiantil', fontsize=16, fontweight='bold')

# 1. Distribución por tipo de movilidad
tipos_movilidad = df['TIPO_MOV_EST_EXTRANJ'].value_counts()
tipos_movilidad.plot(kind='bar', ax=axes[0, 0], color='mediumseagreen')
axes[0, 0].set_title('Distribución por Tipo de Movilidad', fontweight='bold')
axes[0, 0].set_xlabel('Tipo de Movilidad')
axes[0, 0].set_ylabel('Cantidad de Estudiantes')
axes[0, 0].tick_params(axis='x', rotation=45)

# Agregar valores en las barras
for i, v in enumerate(tipos_movilidad.values):
    axes[0, 0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# 2. Distribución de duración por categoría
categoria_duracion = df['CATEGORIA_DURACION'].value_counts()
axes[0, 1].pie(categoria_duracion.values, labels=categoria_duracion.index, 
               autopct='%1.1f%%', startangle=90)
axes[0, 1].set_title('Distribución por Categoría de Duración', fontweight='bold')

# 3. Boxplot de duración por tipo de movilidad
df.boxplot(column='NUM_DIAS_MOVILIDAD', by='TIPO_MOV_EST_EXTRANJ', ax=axes[1, 0])
axes[1, 0].set_title('Distribución de Duración por Tipo de Movilidad', fontweight='bold')
axes[1, 0].set_xlabel('Tipo de Movilidad')
axes[1, 0].set_ylabel('Días de Movilidad')
plt.sca(axes[1, 0])
plt.xticks(rotation=45, ha='right')

# 4. Evolución temporal por tipo
for tipo in df['TIPO_MOV_EST_EXTRANJ'].unique()[:5]:  # Top 5 tipos
    df_tipo = df[df['TIPO_MOV_EST_EXTRANJ'] == tipo]
    evolucion = df_tipo.groupby('AÑO').size()
    axes[1, 1].plot(evolucion.index, evolucion.values, marker='o', label=tipo, linewidth=2)

axes[1, 1].set_title('Evolución Temporal por Tipo de Movilidad', fontweight='bold')
axes[1, 1].set_xlabel('Año')
axes[1, 1].set_ylabel('Número de Estudiantes')
axes[1, 1].legend(loc='best', fontsize=9)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / '03_analisis_tipo_movilidad.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Gráfica guardada: {OUTPUTS_DIR / '03_analisis_tipo_movilidad.png'}")
plt.show()

print("\nESTADÍSTICAS POR TIPO DE MOVILIDAD:")
print("-" * 80)
for tipo, cantidad in tipos_movilidad.items():
    porcentaje = (cantidad / len(df)) * 100
    duracion_prom = df[df['TIPO_MOV_EST_EXTRANJ'] == tipo]['NUM_DIAS_MOVILIDAD'].mean()
    print(f"{tipo}: {cantidad} ({porcentaje:.1f}%) - Duración promedio: {duracion_prom:.1f} días")

---
## 7. ANÁLISIS FINANCIERO

In [ ]:
print("\n" + "="*80)
print("ANÁLISIS FINANCIERO")
print("="*80)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Análisis Financiero de Movilidad Estudiantil', fontsize=16, fontweight='bold')

# 1. Distribución de financiación total
df_fin = df[df['FINANCIACION_TOTAL'] > 0]
axes[0, 0].hist(df_fin['FINANCIACION_TOTAL'], bins=30, color='gold', edgecolor='black')
axes[0, 0].axvline(df_fin['FINANCIACION_TOTAL'].mean(), color='red', linestyle='--', 
                   linewidth=2, label=f"Media: ${df_fin['FINANCIACION_TOTAL'].mean():,.0f}")
axes[0, 0].axvline(df_fin['FINANCIACION_TOTAL'].median(), color='blue', linestyle='--', 
                   linewidth=2, label=f"Mediana: ${df_fin['FINANCIACION_TOTAL'].median():,.0f}")
axes[0, 0].set_title('Distribución de Financiación Total', fontweight='bold')
axes[0, 0].set_xlabel('Financiación (COP)')
axes[0, 0].set_ylabel('Frecuencia')
axes[0, 0].legend()

# 2. Fuentes de financiación
fuentes_financiacion = df['FUENTE_INTERNACIONAL'].value_counts().head(10)
fuentes_financiacion.plot(kind='barh', ax=axes[0, 1], color='lightcoral')
axes[0, 1].set_title('Top 10 Fuentes de Financiación Internacional', fontweight='bold')
axes[0, 1].set_xlabel('Cantidad de Estudiantes')
axes[0, 1].set_ylabel('Fuente')

# 3. Financiación promedio por país
fin_por_pais = df.groupby('PAIS_EXTRANJERO')['FINANCIACION_TOTAL'].mean().nlargest(10)
fin_por_pais.plot(kind='bar', ax=axes[1, 0], color='purple')
axes[1, 0].set_title('Financiación Promedio por País (Top 10)', fontweight='bold')
axes[1, 0].set_xlabel('País')
axes[1, 0].set_ylabel('Financiación Promedio (COP)')
axes[1, 0].tick_params(axis='x', rotation=45)

# 4. Evolución de financiación total anual
fin_anual = df.groupby('AÑO')['FINANCIACION_TOTAL'].sum()
axes[1, 1].bar(fin_anual.index, fin_anual.values, color='steelblue')
axes[1, 1].set_title('Evolución de Financiación Total Anual', fontweight='bold')
axes[1, 1].set_xlabel('Año')
axes[1, 1].set_ylabel('Financiación Total (COP)')

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / '03_analisis_financiero.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Gráfica guardada: {OUTPUTS_DIR / '03_analisis_financiero.png'}")
plt.show()

print("\nESTADÍSTICAS FINANCIERAS:")
print("-" * 80)
print(f"Financiación total registrada: ${df['FINANCIACION_TOTAL'].sum():,.0f} COP")
print(f"Financiación promedio por estudiante: ${df['FINANCIACION_TOTAL'].mean():,.0f} COP")
print(f"Financiación mediana: ${df['FINANCIACION_TOTAL'].median():,.0f} COP")
print(f"Rango de financiación: ${df['FINANCIACION_TOTAL'].min():,.0f} - ${df['FINANCIACION_TOTAL'].max():,.0f} COP")

---
## 8. IDENTIFICACIÓN DE OUTLIERS

In [ ]:
print("\n" + "="*80)
print("IDENTIFICACIÓN DE OUTLIERS")
print("="*80)

def detectar_outliers_iqr(serie, nombre):
    """Detecta outliers usando el método IQR."""
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    outliers = serie[(serie < limite_inferior) | (serie > limite_superior)]
    
    print(f"\n{nombre}:")
    print(f"  - Q1: {Q1:.2f}")
    print(f"  - Q3: {Q3:.2f}")
    print(f"  - IQR: {IQR:.2f}")
    print(f"  - Límites: [{limite_inferior:.2f}, {limite_superior:.2f}]")
    print(f"  - Outliers detectados: {len(outliers)} ({len(outliers)/len(serie)*100:.2f}%)")
    
    return outliers

# Detectar outliers en duración
outliers_duracion = detectar_outliers_iqr(df['NUM_DIAS_MOVILIDAD'].dropna(), "Duración de Movilidad")

# Detectar outliers en financiación
outliers_financiacion = detectar_outliers_iqr(
    df[df['FINANCIACION_TOTAL'] > 0]['FINANCIACION_TOTAL'], 
    "Financiación Total"
)

# Visualización de outliers
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Detección de Outliers', fontsize=16, fontweight='bold')

# Boxplot duración
axes[0].boxplot(df['NUM_DIAS_MOVILIDAD'].dropna(), vert=True)
axes[0].set_title('Outliers en Duración de Movilidad', fontweight='bold')
axes[0].set_ylabel('Días')
axes[0].grid(True, alpha=0.3, axis='y')

# Boxplot financiación
axes[1].boxplot(df[df['FINANCIACION_TOTAL'] > 0]['FINANCIACION_TOTAL'], vert=True)
axes[1].set_title('Outliers en Financiación Total', fontweight='bold')
axes[1].set_ylabel('COP')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / '03_outliers.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Gráfica guardada: {OUTPUTS_DIR / '03_outliers.png'}")
plt.show()

---
## 9. MATRIZ DE CORRELACIONES

In [ ]:
print("\n" + "="*80)
print("ANÁLISIS DE CORRELACIONES")
print("="*80)

# Seleccionar variables numéricas para correlación
vars_numericas = ['AÑO', 'SEMESTRE', 'NUM_DIAS_MOVILIDAD', 'FINANCIACION_TOTAL']
df_corr = df[vars_numericas].dropna()

# Calcular matriz de correlación
corr_matrix = df_corr.corr()

# Visualizar matriz de correlación
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlaciones - Variables Numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / '03_correlaciones.png', dpi=300, bbox_inches='tight')
print(f"\n✓ Gráfica guardada: {OUTPUTS_DIR / '03_correlaciones.png'}")
plt.show()

print("\nMATRIZ DE CORRELACIONES:")
print("-" * 80)
display(corr_matrix)

---
## 10. EXPORTACIÓN DE ESTADÍSTICAS DESCRIPTIVAS

In [ ]:
# Crear DataFrame con estadísticas principales
estadisticas_export = pd.DataFrame({
    'Métrica': [
        'Total Estudiantes',
        'Países Representados',
        'Duración Promedio (días)',
        'Duración Mediana (días)',
        'Financiación Total (COP)',
        'Financiación Promedio (COP)',
        'País Principal',
        'Tipo Movilidad Principal'
    ],
    'Valor': [
        len(df),
        df['PAIS_EXTRANJERO'].nunique(),
        df['NUM_DIAS_MOVILIDAD'].mean(),
        df['NUM_DIAS_MOVILIDAD'].median(),
        df['FINANCIACION_TOTAL'].sum(),
        df['FINANCIACION_TOTAL'].mean(),
        df['PAIS_EXTRANJERO'].value_counts().index[0],
        df['TIPO_MOV_EST_EXTRANJ'].value_counts().index[0]
    ]
})

# Guardar estadísticas
estadisticas_export.to_csv(RESULTS_DIR / 'estadisticas_descriptivas.csv', index=False)
print(f"\n✓ Estadísticas exportadas: {RESULTS_DIR / 'estadisticas_descriptivas.csv'}")

display(estadisticas_export)

---
## 11. SÍNTESIS DE HALLAZGOS

### Hallazgos Principales del Análisis Descriptivo:

1. **Volumen de Movilidad:**
   - Se registraron estudiantes internacionales de múltiples países
   - La distribución geográfica muestra concentración en países específicos

2. **Tendencias Temporales:**
   - Existe variación en la movilidad entre años y semestres
   - Se identifican patrones estacionales en la llegada de estudiantes

3. **Características de la Movilidad:**
   - Duración promedio de estadías varía según país de origen
   - Predominan ciertos tipos de movilidad académica

4. **Aspectos Financieros:**
   - Existe financiación internacional significativa
   - La distribución de recursos muestra variabilidad importante

5. **Calidad de Datos:**
   - Se identificaron outliers que requieren atención
   - Las correlaciones sugieren relaciones entre variables clave

### Próximos Pasos:
- Desarrollar modelos predictivos para proyectar flujos futuros
- Estimar impacto económico basado en patrones identificados
- Crear dashboards interactivos para visualización ejecutiva

---
**Fin del Notebook 3**

Continuar con: `04_modelos_predictivos.ipynb`